# Diabetic Retinopathy Classification (APTOS-style)

End-to-end offline evaluation notebook for the MLOps summative.

**Dataset:** APTOS 2019 fundus images resized to 224×224 (Kaggle Gaussian-filtered variant equivalent). We apply Gaussian blur in preprocessing to match [sovitrath/diabetic-retinopathy-224x224-gaussian-filtered](https://www.kaggle.com/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered).

**Model:** MobileNetV2 (ImageNet) + custom head — transfer learning, dropout, L2, Adam, early stopping.

**Metrics:** Accuracy, Loss, Precision, Recall, F1 (+ confusion matrix).



In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tensorflow import keras

from src.config import CLASS_NAMES, TRAIN_DIR, FULL_DIR, MODEL_PATH, SEED
from src.preprocessing import preprocess_image, image_stats
from src.model import build_model, compile_model, save_model, unfreeze_top
from src.insights import compute_insights

print('ROOT', ROOT)
print('TRAIN_DIR exists', TRAIN_DIR.exists())
print('FULL_DIR exists', FULL_DIR.exists())



## 1. Data acquisition & preprocessing

Steps:
1. Load labeled fundus images from class folders
2. Resize to 224×224
3. Apply Gaussian filter (matches Kaggle Gaussian-filtered APTOS set)
4. Scale pixels to [0, 1]
5. Stratified train/validation split
6. Class weights for imbalance



In [ ]:
def load_dir(root: Path, max_per_class=None):
    xs, ys, paths = [], [], []
    for class_id, name in CLASS_NAMES.items():
        folder = root / name
        files = sorted([p for p in folder.glob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'}])
        if max_per_class:
            files = files[:max_per_class]
        for p in files:
            xs.append(preprocess_image(p, gaussian=True))
            ys.append(class_id)
            paths.append(p)
    return np.stack(xs), np.array(ys), paths

data_root = FULL_DIR if (FULL_DIR / 'No_DR').exists() else TRAIN_DIR
X, y, paths = load_dir(data_root, max_per_class=None)
print('samples', len(y), 'shape', X.shape)
print(Counter(y.tolist()))
pd.Series({CLASS_NAMES[k]: v for k,v in Counter(y.tolist()).items()}).plot(kind='bar', title='Class distribution')
plt.ylabel('count'); plt.show()



In [ ]:
# Feature peek: brightness / green / sharpness by class (feeds UI insights)
rows = []
for class_id, name in CLASS_NAMES.items():
    folder = data_root / name
    for p in list(folder.glob('*'))[:30]:
        s = image_stats(p)
        s['class'] = name
        rows.append(s)
feat = pd.DataFrame(rows)
display(feat.groupby('class')[['brightness','green_mean','sharpness']].mean())
sns.boxplot(data=feat, x='class', y='green_mean')
plt.title('Green-channel intensity by DR grade'); plt.xticks(rotation=30); plt.show()



In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
counts = Counter(y_train.tolist())
n_classes = len(counts)
total = len(y_train)
class_weight = {c: total / (n_classes * n) for c, n in counts.items()}
print('class_weight', class_weight)



## 2. Model creation (transfer learning + optimization)

- Pretrained MobileNetV2 backbone (ImageNet)
- Custom dense head with **Dropout** + **L2** regularization
- Optimizer: **Adam**
- Phase 1: frozen backbone
- Phase 2: fine-tune top layers with lower LR
- **EarlyStopping** + ReduceLROnPlateau



In [ ]:
model = build_model(trainable_base=False, dropout=0.3, l2=1e-4)
compile_model(model, learning_rate=1e-3)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

hist1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

unfreeze_top(model, n_layers=40)
compile_model(model, learning_rate=1e-5)
hist2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

save_model(model, MODEL_PATH)
print('saved', MODEL_PATH)



## 3. Model testing / evaluation (≥4 metrics)



In [ ]:
probs = model.predict(X_val, verbose=0)
y_pred = probs.argmax(axis=1)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='macro', zero_division=0)
recall = recall_score(y_val, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
val_loss = float(model.evaluate(X_val, y_val, verbose=0)[0])

metrics = {
    'Accuracy': accuracy,
    'Loss': val_loss,
    'Precision (macro)': precision,
    'Recall (macro)': recall,
    'F1 (macro)': f1,
}
display(pd.Series(metrics))
print(classification_report(y_val, y_pred, target_names=[CLASS_NAMES[i] for i in range(5)], zero_division=0))

cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[CLASS_NAMES[i] for i in range(5)],
            yticklabels=[CLASS_NAMES[i] for i in range(5)])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix'); plt.show()

# Training curves
hist = {k: hist1.history.get(k, []) + hist2.history.get(k, []) for k in ['loss','val_loss','accuracy','val_accuracy']}
pd.DataFrame(hist).plot(title='Training curves'); plt.show()

compute_insights(TRAIN_DIR if TRAIN_DIR.exists() else data_root)
print('insights cached for UI')



## 4. Prediction helper (single image)

Mirrors `src/prediction.py` used by the API.



In [ ]:
from src.prediction import predict_image, clear_model_cache
clear_model_cache()
sample = paths[0]
print('sample', sample)
print(predict_image(sample))

